# Phase 17 — Model Merging & Export
## CodeMentor-LLM
Merging DPO LoRA adapter into base model weights
for deployment-ready model.

## Goal
- Load DPO model with LoRA adapter
- Merge adapter into base weights using merge_and_unload()
- Push merged model to HuggingFace Hub
- Test merged model inference

# libraries

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes==0.45.3 accelerate==1.5.1

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load DPO model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
dpo_adapter_id = "Abdulmoiz123/codementor-llm-dpo"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Load DPO adapter
print("Loading DPO adapter...")
model = PeftModel.from_pretrained(base_model, dpo_adapter_id)
print(f"Model loaded — {model.get_memory_footprint() / 1024**3:.2f} GB")

# Merge adapter into base model

In [ ]:
# Merge LoRA adapter into base model weights
print("Merging adapter into base model...")

merged_model = model.merge_and_unload()
print("Adapter merged successfully")
print(f"Merged model type: {type(merged_model)}")
print(f"Memory footprint: {merged_model.get_memory_footprint() / 1024**3:.2f} GB")

#  Test merged model inference

In [ ]:
SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def generate_response(model, tokenizer, instruction, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

# Test with 3 coding questions
test_questions = [
    "Write a Python function to reverse a string.",
    "What is the difference between a list and a tuple in Python?",
    "Write a SQL query to find duplicate records in a table."
]

for i, question in enumerate(test_questions):
    print(f"\nQ{i+1}: {question}")
    print(f"A: {generate_response(merged_model, tokenizer, question)}")
    print("=" * 50)

# Push merged model to HuggingFace Hub

In [ ]:
# Push merged model to HF Hub
print("Pushing merged model to HuggingFace Hub...")

merged_model.push_to_hub("Abdulmoiz123/codementor-llm-merged")
tokenizer.push_to_hub("Abdulmoiz123/codementor-llm-merged")
print("Merged model pushed successfully")
print(f"Model: https://huggingface.co/Abdulmoiz123/codementor-llm-merged")